# Global Savings Glut: Current Account Balances

The "global savings glut" hypothesis (Bernanke 2005) argues that excess savings
over investment in key economies drove capital flows into deficit countries,
depressing global interest rates.

**Data source**: World Bank Development Indicators
- `BN.CAB.XOKA.GD.ZS` — Current account balance (% of GDP)

**Coverage**: G20 nations plus G10 members (Netherlands, Belgium, Sweden, Switzerland) and Spain.

In [1]:
# system imports
from io import StringIO

# analytic imports
import matplotlib.pyplot as plt
import pandas as pd

# local imports
import common
import mgplot as mg

In [2]:
# chart setup
CHART_DIR = "./CHARTS/Savings-Glut-World-Bank/"
mg.set_chart_dir(CHART_DIR)
mg.clear_chart_dir()
SHOW = False
SOURCE = "World Bank"

## Fetch current account balance data from World Bank

In [3]:
# Major economy country codes
COUNTRY_CODES = (
    "USA;GBR;DEU;FRA;ITA;JPN;CAN;"  # G7
    "NLD;BEL;SWE;CHE;"               # additional G10/G11
    "AUS;CHN;IND;KOR;IDN;SAU;"       # Asia & Middle East
    "BRA;MEX;ARG;"                    # Americas
    "ZAF;RUS;TUR;"                    # South Africa, Russia, Türkiye
    "ESP"                             # Spain
)


def get_wb_indicator(
    indicator: str,
    countries: str = COUNTRY_CODES,
    start: int = 1980,
    end: int = 2024,
) -> pd.DataFrame:
    """Fetch a World Bank indicator via JSON API and return as a pivoted
    DataFrame with years as index and country names as columns."""

    import json

    page = 1
    records = []
    while True:
        url = (
            f"https://api.worldbank.org/v2/country/{countries}"
            f"/indicator/{indicator}"
            f"?format=json&date={start}:{end}"
            f"&per_page=1000&page={page}"
        )
        raw = common.request_get(url)
        data = json.loads(raw)
        if len(data) < 2 or data[1] is None:
            break
        for row in data[1]:
            if row["value"] is not None:
                records.append({
                    "country": row["country"]["value"],
                    "year": int(row["date"]),
                    "value": float(row["value"]),
                })
        total_pages = data[0]["pages"]
        if page >= total_pages:
            break
        page += 1

    df = pd.DataFrame(records)
    result = df.pivot(index="year", columns="country", values="value")
    result.index = pd.PeriodIndex(result.index.astype(str), freq="Y")
    result = result.sort_index()
    result = result.dropna(how="all", axis=0).dropna(how="all", axis=1)
    return result


cab = get_wb_indicator("BN.CAB.XOKA.GD.ZS")
print(f"Nations ({len(cab.columns)}): {list(cab.columns)}")
cab.tail()

Nations (24): ['Argentina', 'Australia', 'Belgium', 'Brazil', 'Canada', 'China', 'France', 'Germany', 'India', 'Indonesia', 'Italy', 'Japan', 'Korea, Rep.', 'Mexico', 'Netherlands', 'Russian Federation', 'Saudi Arabia', 'South Africa', 'Spain', 'Sweden', 'Switzerland', 'Turkiye', 'United Kingdom', 'United States']


country,Argentina,Australia,Belgium,Brazil,Canada,China,France,Germany,India,Indonesia,...,Netherlands,Russian Federation,Saudi Arabia,South Africa,Spain,Sweden,Switzerland,Turkiye,United Kingdom,United States
year,,,,,,,,,,,,,,,,,,,,,
2020,0.696935,1.867418,0.903941,-1.641035,-2.010494,1.659301,-2.003136,6.347827,1.223621,-0.418606,...,5.722964,2.369115,-4.516921,2.003523,0.819280,5.606152,0.447365,-4.222302,-2.928834,-2.818094
2021,1.361544,2.625448,1.868309,-2.358221,-0.021963,1.938752,0.244239,6.920070,-1.055242,0.295886,...,10.278137,6.831070,4.569763,3.772414,0.765005,6.219752,7.671531,-0.740649,-0.752836,-3.682741
2022,-0.625255,0.374774,-1.821515,-2.149677,-0.288416,2.420593,-1.432349,3.850456,-2.362475,1.001829,...,6.821477,10.374128,11.729451,-0.296183,0.379418,4.011732,9.283212,-4.997638,-2.067371,-3.878726
2023,-3.195146,-0.234061,0.182293,-1.235582,-0.633320,1.441583,-1.043017,5.512225,-0.878263,-0.148899,...,9.414092,2.384567,2.129368,-1.103563,2.732461,5.810826,5.899766,-3.494173,-3.576762,-3.400305
2024,0.893118,-1.978112,-0.378788,-3.027160,-0.461254,2.261650,0.085128,5.771885,-0.822243,-0.624744,...,9.129968,2.914680,-1.311018,-0.642800,3.183274,5.924622,7.689099,-0.749968,-2.187936,-4.122625


## Classify nations by current account position

In [4]:
# Classify based on average current account balance
THRESHOLD = 1.5  # % of GDP - above = surplus, below negative = deficit, else near-balance
CLASSIFICATION_YEARS = 20

recent = cab.iloc[-CLASSIFICATION_YEARS:]
avg = recent.mean().sort_values(ascending=False)

surplus_nations = avg[avg > THRESHOLD].index.tolist()
deficit_nations = avg[avg < -THRESHOLD].index.tolist()
balance_nations = avg[(avg >= -THRESHOLD) & (avg <= THRESHOLD)].index.tolist()

print(f"Surplus (>{THRESHOLD}% avg):  {surplus_nations}")
print(f"Near-balance:         {balance_nations}")
print(f"Deficit (<-{THRESHOLD}% avg): {deficit_nations}")
print()
display(avg.to_frame(f"{CLASSIFICATION_YEARS}yr avg CA (% GDP)").round(1))

Surplus (>1.5% avg):  ['Saudi Arabia', 'Switzerland', 'Netherlands', 'Germany', 'Sweden', 'Russian Federation', 'Korea, Rep.', 'China', 'Japan']
Near-balance:         ['Belgium', 'Italy', 'Indonesia', 'Argentina', 'France', 'Spain', 'Mexico']
Deficit (<-1.5% avg): ['Canada', 'India', 'Brazil', 'South Africa', 'Australia', 'United Kingdom', 'United States', 'Turkiye']



,20yr avg CA (% GDP)
country,
Saudi Arabia,10.7
Switzerland,7.6
Netherlands,7.1
Germany,6.6
Sweden,5.2
Russian Federation,4.9
"Korea, Rep.",3.3
China,3.3
Japan,3.1


## Surplus nations: the savings glut producers

In [5]:
PLOT_KWARGS = dict(
    ylabel="Per cent of GDP",
    width=2,
    y0=True,
    rfooter=SOURCE,
    lfooter="Current account balance as percentage of GDP. "
    "Positive = net capital exporter.",
    legend={"loc": "best", "fontsize": "x-small", "ncol": 2},
    show=SHOW,
)

mg.line_plot_finalise(
    cab[surplus_nations].dropna(how="all"),
    title="Current account surplus nations",
    **PLOT_KWARGS,
)

## Near-balance nations

In [6]:
mg.line_plot_finalise(
    cab[balance_nations].dropna(how="all"),
    title="Current account near-balance nations",
    **PLOT_KWARGS,
)

## Deficit nations: the capital importers

In [7]:
mg.line_plot_finalise(
    cab[deficit_nations].dropna(how="all"),
    title="Current account deficit nations",
    **PLOT_KWARGS,
)

## The savings glut in proportion: group totals as % of world GDP

Expressing each group's combined current account balance as a share of
world GDP reveals the true scale of the imbalances — because a 2% surplus
on a fast-growing economy like China produces far more capital than
the percentage alone suggests.

In [8]:
# Fetch current account in USD and world GDP
cab_usd = get_wb_indicator("BN.CAB.XOKA.CD")  # current account balance, current USD
world_gdp = get_wb_indicator("NY.GDP.MKTP.CD", countries="WLD")  # world GDP, current USD

print(f"CA USD nations: {len(cab_usd.columns)}")
print(f"World GDP available: {world_gdp.index.min()} to {world_gdp.index.max()}")
display(cab_usd.tail())

CA USD nations: 24
World GDP available: 1980 to 2024


country,Argentina,Australia,Belgium,Brazil,Canada,China,France,Germany,India,Indonesia,...,Netherlands,Russian Federation,Saudi Arabia,South Africa,Spain,Sweden,Switzerland,Turkiye,United Kingdom,United States
year,,,,,,,,,,,,,,,,,,,,,
2020,2.688362e+09,2.489896e+10,4.788128e+09,-2.422344e+10,-3.328743e+10,2.488356e+11,-5.304156e+10,2.501932e+11,3.273005e+10,-4.433270e+09,...,5.337012e+10,3.537269e+10,-3.468775e+10,6.771399e+09,1.056694e+10,3.051236e+10,3.319443e+09,-3.097600e+10,-7.978148e+10,-5.935040e+11
2021,6.624785e+09,4.097320e+10,1.118225e+10,-3.939756e+10,-4.441797e+08,3.528858e+11,7.245174e+09,3.013865e+11,-3.342236e+10,3.510715e+09,...,1.083801e+11,1.249530e+11,4.490528e+10,1.584362e+10,1.117859e+10,3.928976e+10,6.254671e+10,-6.221000e+09,-2.404980e+10,-8.586340e+11
2022,-3.964075e+09,6.354771e+09,-1.076672e+10,-4.196006e+10,-6.317507e+09,4.433743e+11,-4.003111e+10,1.617585e+11,-7.905094e+10,1.321513e+10,...,7.138954e+10,2.377348e+11,1.453367e+11,-1.207229e+09,5.497205e+09,2.307031e+10,7.691224e+10,-4.628300e+10,-6.576812e+10,-9.931420e+11
2023,-2.075125e+10,-4.059666e+09,1.187333e+09,-2.707323e+10,-1.376419e+10,2.633824e+11,-3.187721e+10,2.514791e+11,-3.195550e+10,-2.041662e+09,...,1.068947e+11,4.939645e+10,2.594815e+10,-4.209439e+09,4.425171e+10,3.364416e+10,5.276897e+10,-3.987700e+10,-1.223537e+11,-9.280170e+11
2024,5.701359e+09,-3.475587e+10,-2.543070e+09,-6.616831e+10,-1.034886e+10,4.239192e+11,2.690421e+09,2.704470e+11,-3.214882e+10,-8.723308e+09,...,1.109225e+11,6.336035e+10,-1.625406e+10,-2.578559e+09,5.493286e+10,3.576784e+10,7.201335e+10,-1.019300e+10,-8.064804e+10,-1.185294e+12


In [9]:
# Sum positive and negative current accounts as % of world GDP
wgdp = world_gdp.iloc[:, 0]  # Series

# For each year, split countries into positive and negative CA balances
ca_pct_of_wgdp = cab_usd.div(wgdp, axis=0) * 100

pos_neg = pd.DataFrame(index=ca_pct_of_wgdp.index)
pos_neg["Sum of surpluses"] = ca_pct_of_wgdp.clip(lower=0).sum(axis=1)
pos_neg["Sum of deficits"] = ca_pct_of_wgdp.clip(upper=0).sum(axis=1)
pos_neg = pos_neg.dropna(how="all")

# Build nation list text box (4 rows of ~6 names)
names = sorted(cab_usd.columns)
n_per_row = 6
nation_lines = []
for i in range(0, len(names), n_per_row):
    nation_lines.append(", ".join(names[i:i + n_per_row]))
NATION_TEXT = "\n".join(nation_lines)

ax = mg.line_plot(
    pos_neg,
    width=2.5,
)
ax.text(
    0.02, 0.02, NATION_TEXT,
    transform=ax.transAxes,
    fontsize=6, ha="left", va="bottom",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8, edgecolor="lightgray"),
)
mg.finalise_plot(
    ax,
    title="Current account surpluses and deficits\nas share of world GDP",
    ylabel="Per cent of world GDP",
    y0=True,
    rfooter=SOURCE,
    lfooter="Each year: sum of all positive (negative) current account balances (USD) "
    "divided by world GDP.",
    legend={"loc": "best", "fontsize": "small"},
    show=SHOW,
)

In [10]:
# Net current account balance as % of world GDP (surplus + deficit)
net_ca = (pos_neg["Sum of surpluses"] + pos_neg["Sum of deficits"]).to_frame(
    "Net current account"
)

ax = mg.line_plot(
    net_ca,
    width=2.5,
)
ax.text(
    0.02, 0.98, NATION_TEXT,
    transform=ax.transAxes,
    fontsize=6, ha="left", va="top",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8, edgecolor="lightgray"),
)
mg.finalise_plot(
    ax,
    title="Net current account balance as share of world GDP",
    ylabel="Per cent of world GDP",
    y0=True,
    rfooter=SOURCE,
    lfooter="Sum of all current account balances (USD) divided by world GDP.",
    legend=False,
    show=SHOW,
)

## The End

In [11]:
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-03-14 10:44:25

Python implementation: CPython
Python version       : 3.14.0
IPython version      : 9.10.0

conda environment: n/a

Compiler    : Clang 20.1.4 
OS          : Darwin
Release     : 25.3.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

matplotlib: 3.10.8
mgplot    : 0.2.19
pandas    : 3.0.1

Watermark: 2.6.0

